# Exploratory Data Analysis - Suspect Location ML

This notebook covers raw GPS exploration, clustered place analysis, temporal behavior, and sample model predictions.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from branca.colormap import linear
from IPython.display import display

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

RAW_PATH = Path('data/raw/gps_log.csv')
PREDICT_SCRIPT = Path('src/predict.py')
PRED_OUT_DIR = Path('outputs/predictions')
KNOWN_LOCATIONS = {
    'home': (19.1136, 72.8697),
    'office': (19.0176, 72.8562),
    'gym': (19.1121, 72.8711),
    'local_market': (19.1150, 72.8670),
    'mall': (19.0883, 72.8276),
    'friend_area': (19.0595, 72.8307),
}


## 1) Load Raw GPS Data, Show Shape and dtypes

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f'Raw data not found: {RAW_PATH}')

df = pd.read_csv(RAW_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

print('Shape:', df.shape)
display(df.head())
display(df.dtypes.to_frame('dtype'))


## 2) Folium Map - GPS Points Colored by Hour of Day

In [ ]:
plot_df = df.dropna(subset=['latitude', 'longitude', 'hour']).copy()
max_points = 3000
if len(plot_df) > max_points:
    plot_df = plot_df.sample(max_points, random_state=42).sort_values('timestamp')

center = [plot_df['latitude'].mean(), plot_df['longitude'].mean()]
m_hour = folium.Map(location=center, zoom_start=12, tiles='CartoDB positron')
hour_cmap = linear.viridis.scale(0, 23)

for row in plot_df.itertuples(index=False):
    color = hour_cmap(float(row.hour))
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=2,
        color=color,
        fill=True,
        fill_opacity=0.6,
        weight=0,
        popup=f"{row.timestamp} | hour={int(row.hour)}",
    ).add_to(m_hour)

hour_cmap.caption = 'Hour of Day'
hour_cmap.add_to(m_hour)
m_hour


## 3) Folium Map - GPS Points Colored by Day of Week

In [ ]:
plot_df2 = df.dropna(subset=['latitude', 'longitude', 'day_of_week']).copy()
max_points = 3000
if len(plot_df2) > max_points:
    plot_df2 = plot_df2.sample(max_points, random_state=7).sort_values('timestamp')

dow_colors = {
    0: 'red',
    1: 'blue',
    2: 'green',
    3: 'orange',
    4: 'purple',
    5: 'brown',
    6: 'black',
}
dow_names = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}

center2 = [plot_df2['latitude'].mean(), plot_df2['longitude'].mean()]
m_dow = folium.Map(location=center2, zoom_start=12, tiles='CartoDB positron')

for row in plot_df2.itertuples(index=False):
    dow = int(row.day_of_week)
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=2,
        color=dow_colors.get(dow, 'gray'),
        fill=True,
        fill_opacity=0.6,
        weight=0,
        popup=f"{row.timestamp} | dow={dow_names.get(dow, dow)}",
    ).add_to(m_dow)

m_dow


## 4) After Clustering - Cluster Centroids on Map with Labels

In [ ]:
if 'place_id' not in df.columns or 'place_label' not in df.columns:
    raise ValueError('Expected place_id and place_label columns. Run clustering step first.')

cluster_df = df[df['place_id'] != -1].copy()
centroids = (
    cluster_df.groupby('place_id', as_index=False)
    .agg(
        centroid_lat=('latitude', 'mean'),
        centroid_lon=('longitude', 'mean'),
        count=('place_id', 'size'),
        place_label=('place_label', lambda s: s.mode().iat[0] if not s.mode().empty else 'unknown')
    )
    .sort_values('count', ascending=False)
)
display(centroids)

m_centroid = folium.Map(
    location=[df['latitude'].mean(), df['longitude'].mean()],
    zoom_start=12,
    tiles='CartoDB positron',
)

for row in centroids.itertuples(index=False):
    text = f"ID={int(row.place_id)} | {row.place_label} | n={int(row.count)}"
    folium.Marker(
        location=[row.centroid_lat, row.centroid_lon],
        popup=text,
        tooltip=row.place_label,
    ).add_to(m_centroid)

m_centroid


## 5) Time Distribution - Points per place_label (Pie Chart)

In [ ]:
label_counts = df['place_label'].fillna('NaN').value_counts()

plt.figure(figsize=(7, 7))
plt.pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Place Label Distribution')
plt.axis('equal')
plt.show()


## 6) Weekday vs Weekend - Side-by-Side Bar Chart

In [ ]:
week_df = df.dropna(subset=['place_label']).copy()
if 'is_weekend' not in week_df.columns:
    week_df['day_of_week'] = week_df['timestamp'].dt.dayofweek
    week_df['is_weekend'] = week_df['day_of_week'] >= 5

dist = (
    week_df.groupby(['is_weekend', 'place_label'])
    .size()
    .rename('count')
    .reset_index()
)
dist['day_type'] = dist['is_weekend'].map({False: 'Weekday', True: 'Weekend'})

plt.figure(figsize=(11, 6))
sns.barplot(data=dist, x='place_label', y='count', hue='day_type')
plt.title('Place Distribution: Weekday vs Weekend')
plt.xlabel('Place Label')
plt.ylabel('Point Count')
plt.xticks(rotation=20)
plt.show()


## 7) Hourly Heatmap - Hour vs place_label Frequency (Weekday Only)

In [ ]:
hm_df = df.dropna(subset=['place_label']).copy()
if 'hour' not in hm_df.columns:
    hm_df['hour'] = hm_df['timestamp'].dt.hour
if 'day_of_week' not in hm_df.columns:
    hm_df['day_of_week'] = hm_df['timestamp'].dt.dayofweek

weekday = hm_df[hm_df['day_of_week'] < 5]
weekday_pivot = (
    weekday.groupby(['place_label', 'hour'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(24), fill_value=0)
)

plt.figure(figsize=(14, 4.5))
sns.heatmap(weekday_pivot, cmap='Blues')
plt.title('Weekday Hour vs Place Frequency')
plt.xlabel('Hour')
plt.ylabel('Place Label')
plt.show()


## 8) Hourly Heatmap - Hour vs place_label Frequency (Weekend Only)

In [ ]:
weekend = hm_df[hm_df['day_of_week'] >= 5]
weekend_pivot = (
    weekend.groupby(['place_label', 'hour'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(24), fill_value=0)
)

plt.figure(figsize=(14, 4.5))
sns.heatmap(weekend_pivot, cmap='Oranges')
plt.title('Weekend Hour vs Place Frequency')
plt.xlabel('Hour')
plt.ylabel('Place Label')
plt.show()


## 9) Sample Predictions - Run `predict.py` for 3 Dates

In [ ]:
sample_dates = ['2024-06-10', '2024-06-15', '2024-06-22']

if not PREDICT_SCRIPT.exists():
    raise FileNotFoundError(f'Predict script not found: {PREDICT_SCRIPT}')

for d in sample_dates:
    print('=' * 90)
    print(f'Running prediction for {d}')
    result = subprocess.run(
        [sys.executable, str(PREDICT_SCRIPT), '--date', d],
        capture_output=True,
        text=True,
        check=True,
    )
    print(result.stdout)

    pred_json = PRED_OUT_DIR / f'{d}.json'
    if pred_json.exists():
        with pred_json.open('r', encoding='utf-8') as f:
            payload = json.load(f)

        rows = []
        for hour_key, model_probs in payload['hourly'].items():
            rf = model_probs.get('rf', {})
            if not rf:
                rows.append((hour_key, 'n/a', 0.0))
                continue
            top_label = max(rf.items(), key=lambda kv: kv[1])[0]
            top_prob = rf[top_label]
            rows.append((hour_key, top_label, top_prob))

        summary = pd.DataFrame(rows, columns=['hour', 'rf_top_label', 'rf_prob'])
        display(summary.head(10))
